In [6]:
import ast
import optuna
import ipynbname
import pandas as pd
import numpy as np
from optuna.exceptions import TrialPruned
from Functions.AutoCloud_V3 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT import *

FileName = ipynbname.name()
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = HI['PC1'].values

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset\\Bearing1_1.csv'

In [ ]:
params =  [2.0, 5, [58, 11, 64, 31, 32], 15, 5, [1,1,1], 10,'ahead', 'tanh']

m,nI,nR,nO,mO,ηS,τ,mode,act = params
if isinstance(nR, str): nR = ast.literal_eval(nR)
X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)

teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=ηS,mode=mode,act=act,
               tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=False,wtaP=True) 

for j,_ in enumerate(X[:]):
    
    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
    teda.WAPE_HI2(Y[j],Z[j])
    teda.Adapt(Y[j],Z[j])
    

#teda.c = np.append(teda.c,teda.gm)  
PlotDSI_3D_PLT(teda)


In [ ]:
print(xxxx)

In [ ]:
def Optimize(
        FileName=None,OptDim=None,OptSampler=None,OptSeed=None,OptPrune=False,
        n_study=None,timeout=None,n_trials=None,
        patience=None,
        mS =None,nIS=None,nLS=None,nRS=None,nOS =None,mOS=None,
        N1S=None,N2S=None,N3S=None,τS =None,mdS=None,actS=None):

    '''
        OptSampler modes available: None, Auto, GP, NSGAII, Random, TPE
    
        None: sampler default, suport multivariate optimization

        GP: suport multivariate optimization

        Auto: suport multivariate optimization
        
        NSGAII: poorly suport multivariate optimization
        
        Random: poorly suport multivariate optimization

        TPE: suport multivariate optimization
    '''

    if OptDim == 1 or OptDim is None: SingleObj = True
    elif OptDim > 1: SingleObj = False    
    if n_study is None: n_study = 1
    if timeout is None: timeout = 60
    if n_trials is None and patience is None: 
        n_trials = 25
        patience = 25
    elif n_trials < 100 and patience is None:
        patience = int(0.25*n_trials)
    elif n_trials >= 1e2 and n_trials < 1e3 and patience is None:
        patience = int(0.20*n_trials)
    elif n_trials >= 1e3 and n_trials < 5e3 and patience is None:
        patience = int(0.150*n_trials)
    elif n_trials >= 5e3 and n_trials < 1e4 and patience is None:
        patience = int(0.125*n_trials)    
    elif n_trials >= 1e4 and patience is None:
        patience = int(0.03*n_trials)   

    if mS  is None: mS  = [1.75,4.25]
    if nIS is None: nIS = [2,40]
    if nLS is None: nLS = [1,5]
    if nRS is None: nRS = [1,60]
    if nOS is None: nOS = [1,40]
    if mOS is None: mOS = [0,40]
    if N1S is None: N1S = [1,9]
    if N2S is None: N2S = [1,9]
    if N3S is None: N3S = [1,9]
    if τS  is None: τS  = [1,25]
    if mdS is None: mdS = [0,1]
    if actS is None: actS = [0,1,2]

    if SingleObj: names = ['MAPE_RUL+MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N1', 'N2', 'N3','TAU','past/ahead','activation']
    else: names = ['MAPE_RUL','MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N1', 'N2', 'N3','TAU','past/ahead','activation']
    study_dir, out_path = df_ParamsTable(names,FileName)

    for i in range(n_study):
        print('iteration:',i+1)
        def objective(trial):
            
            m = trial.suggest_float('m', mS[0], mS[1],step=0.25)
            τ = trial.suggest_int('τ', τS[0], τS[1])
            mode = trial.suggest_int('mode', mdS[0], mdS[1])  
            act = trial.suggest_int('act', actS[0], actS[1])
            N1 = trial.suggest_int('N1', N1S[0], N1S[1])
            N2 = trial.suggest_int('N2', N2S[0], N2S[1])
            N3 = trial.suggest_int('N3', N3S[0], N3S[1])
            nI = trial.suggest_int('nI', nIS[0], nIS[1]) 
            nO = trial.suggest_int('nO', nOS[0], nOS[1]) 
            if mode == 'past' or mode == 0:
                mO = trial.suggest_int('mO', 0, 0) 
                if nO > nI: raise TrialPruned()
            elif mode == 'ahead' or mode == 1:
                mO = trial.suggest_int('mO', mOS[0], mOS[1]) 
                if   nI > 20 or nO > 20: raise TrialPruned()
                elif mO > nI: raise TrialPruned()
            nL = trial.suggest_int('nL', nLS[0], nLS[1])
            nR = [trial.suggest_int(f'nL{l}', nRS[0], nRS[1]) for l in range(nL)]

            X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
            
            teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
                           tau=τ,rho=0.0,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
            for j,_ in enumerate(X[:]):
                teda.run(X[j])
                teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1)
                teda.Adapt(Y[j],Z[j])
                #teda.WAPE_HI2(Y[j],Z[j])

                if np.isinf(teda.hiP[-1]) or np.isnan(teda.hiP[-1]): raise TrialPruned()
                if not OptPrune: continue
        
                if j >=60 and j%5==0:
                    if teda.wape_RUL > 0.6: raise TrialPruned()
                    if teda.wape_HI  > 0.3: raise TrialPruned()

                if SingleObj and j%35==0:
                    if np.mean(teda.rulP) == 1: raise TrialPruned()
                    trial.report(teda.wape_RUL+teda.wape_HI, j)
                    if trial.should_prune():
                        raise optuna.TrialPruned()

            if SingleObj: return teda.wape_RUL + teda.wape_HI
            else: return teda.wape_RUL,teda.wape_HI

        pruner=optuna.pruners.HyperbandPruner()
        if SingleObj: study = optuna.create_study(direction='minimize',pruner=pruner,sampler=SelSampler(mode=OptSampler,seed=OptSeed))
        else: study = optuna.create_study(directions=['minimize','minimize'],sampler=SelSampler(mode=OptSampler,seed=OptSeed))
        study.optimize(objective, n_trials=n_trials, timeout=timeout, 
                       callbacks=[EarlyStoppingCallback(patience=patience)]
                       )

        vec = []
        if SingleObj:
            trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
            for trial in trials:
                p = trial.params
                nR_list = [p[f'nL{l}'] for l in range(p['nL'])]
                row = [trial.values[0],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N1'],p['N2'],p['N3'],p['τ'],p['mode'],p['act']]   
                vec.append(row)
            df2 = pd.DataFrame(vec,columns=names)
            df2 = df2.sort_values(by=df2.columns[0], ascending=False)[-5:]

        elif not SingleObj: 
            for trial in study.best_trials:
                p = trial.params
                nR_list = [p[f'nL{l}'] for l in range(p['nL'])]
                row = [trial.values[0],trial.values[1],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N1'],p['N2'],p['N3'],p['τ'],p['mode'],p['act']]
                vec.append(row)
            df2 = pd.DataFrame(vec,columns=names)
        
    if os.path.isfile(out_path) and out_path.startswith(study_dir):
        df1 = pd.read_csv(out_path)
        df_stdy = pd.concat([df1, df2], ignore_index=True)
    
    else: df_stdy = df2
    
    df_stdy = df_stdy.drop_duplicates()
    df_stdy.to_csv(out_path, index=False)

    if not SingleObj: 
        Sum = df_stdy.iloc[:, 0] + df_stdy.iloc[:, 1]
        df_stdy = df_stdy.loc[Sum.sort_values(ascending=False).index]
    elif SingleObj:
        Sum = df_stdy.iloc[:, 0] + 0
        df_stdy = df_stdy.loc[Sum.sort_values(ascending=False).index]
    
    return df_stdy

In [ ]:
df = Optimize(FileName=FileName[:-3],OptDim=2,OptSampler='gp',OptPrune=False,
                         n_study=1,timeout=1200,n_trials=50,patience=1e3,
                         mS=[2.0,4.5],nLS=[1,4],nRS=[1,70],mdS=[1,1],actS=[0,0])

In [ ]:
for i in range(1,6):
    df = Optimize(FileName=FileName[:-3],OptDim=1,OptSampler='tpe',OptPrune=False,
                         n_study=1,timeout=600,n_trials=1e4,patience=1.5e3,
                         mS=[2.0,4.5],nLS=[i,i],nIS=[2,20],nRS=[1,70],mdS=[1,1],actS=[0,0],
                         N1S=[1,4],N2S=[1,9],N3S=[1,8],τS =[1,30])

In [ ]:
params_list = df.values[:,-11:]
df[-10:]

In [ ]:
tedas = []
for i,params in enumerate(params_list[-1:]):
    m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params
    X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
    if isinstance(nR, str): nR = ast.literal_eval(nR)

    teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
                tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
    for j,_ in enumerate(X[:]):
        teda.run(X[j])
        teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
        teda.Adapt(Y[j],Z[j])

    teda.c = np.append(teda.c,teda.gm)  
    tedas.append(teda)
    #PlotSeriesPLY(ySeries=[teda.wape_HI_hist,teda.wape_RUL_hist])

In [ ]:
PlotDSI_3D_PLT(tedas[0])

In [ ]:
PlotDSI_3D_PLT(tedas[0])

# RFLO (Murray, 2019) vs. `Testing/DRTLO_GD_R2.py`

**Sources compared**

- **Paper:** J. M. Murray, *Local online learning in recurrent networks with random feedback*, eLife 8:e43299 (2019), [rflo.pdf](rflo.pdf). Main text eqs. (1)–(7), Appendix 1 eqs. (10)–(17).
- **Authors' code:** [RFLO.ipynb](RFLO.ipynb), class `RNN.run_trial`.
- **Your code:** [Testing/DRTLO_GD_R2.py](../Testing/DRTLO_GD_R2.py), class `RTLO`.

This is the class R07 uses: [eGDRNN_GD_R07.ipynb](../eGDRNN_GD_R07.ipynb) imports `Functions/AutoCloud.py`, which imports `Functions/DataCloud.py`, which imports `Testing/DRTLO_GD_R2.py`. Every granule creates one `RTLO` object (`DataCloud.__init__`).

---

## 1. RFLO as defined in the paper

### 1.1 Model (single recurrent layer, no bias)

$$
u(t) = W\,h(t-1) + W^{in}\,x(t), \qquad
h(t) = h(t-1) + \frac{1}{\tau}\Big[-h(t-1) + \phi\big(u(t)\big)\Big], \qquad
y(t) = W^{out} h(t)
$$

Loss: $\varepsilon(t) = y^*(t) - y(t)$, $\;L = \frac{1}{2T}\sum_t \lVert \varepsilon(t) \rVert^2$, with $\phi = \tanh$.

### 1.2 Exact gradient (RTRL), Appendix 1 eqs. (10)–(11)

$$
\Delta W_{ab} = \frac{\eta_2}{T}\sum_t \sum_j \big[(W^{out})^{\top}\varepsilon(t)\big]_j\, P^j_{ab}(t)
$$

$$
P^j_{ab}(t) = \Big(1-\tfrac{1}{\tau}\Big)P^j_{ab}(t-1)
\;+\; \underbrace{\tfrac{1}{\tau}\sum_k \phi'\big(u_j(t)\big) W_{jk} P^k_{ab}(t-1)}_{\text{non-local term}}
\;+\; \tfrac{1}{\tau}\,\delta_{ja}\,\phi'\big(u_a(t)\big)\,h_b(t-1)
$$

### 1.3 The two RFLO approximations

1. **Locality:** drop the non-local term. The rank-3 tensor $P^j_{ab}$ then collapses to a rank-2 *eligibility trace* $p_{ab}$.
2. **Random feedback:** replace $(W^{out})^{\top}$ with a fixed random matrix $B$ (feedback alignment, Lillicrap et al. 2016).

The resulting rule is the paper's eqs. (5)–(6):

$$
\delta W^{out}_{ab} = \eta_1\,\varepsilon_a(t)\,h_b(t), \qquad
\delta W_{ab} = \eta_2\,[B\varepsilon(t)]_a\,p_{ab}(t), \qquad
\delta W^{in}_{ab} = \eta_3\,[B\varepsilon(t)]_a\,q_{ab}(t)
$$

$$
p_{ab}(t) = \Big(1-\tfrac{1}{\tau}\Big)p_{ab}(t-1) + \tfrac{1}{\tau}\,\phi'\big(u_a(t)\big)\,h_b(t-1), \qquad
q_{ab}(t) = \Big(1-\tfrac{1}{\tau}\Big)q_{ab}(t-1) + \tfrac{1}{\tau}\,\phi'\big(u_a(t)\big)\,x_b(t)
$$

Cost: $\mathcal{O}(N^2)$ per time step, against $\mathcal{O}(N^4)$ for RTRL.

> **Index inconsistency in the paper.** The equations above follow from the model in §1.1 (eq. 13 of the paper: $u(t) = W h(t-1) + W^{in}x(t)$). The paper itself writes the trace three different ways:
>
> - eq. (6): $\phi'(u_a(t))\,x_b(t-1)$
> - eq. (17): $\phi'(u_a(t-1))\,h_b(t-1)$ and $\phi'(u_a(t-1))\,x_b(t-1)$
> - eq. (11): $\phi'(u_a(t))\,x_b(t-1)$
>
> Differentiating eq. (13) gives $\partial u_a(t)/\partial W^{in}_{ab} = x_b(t)$ and $\partial u_a(t)/\partial W_{ab} = h_b(t-1)$. So the forms above are the correct ones, and the paper's variants are notation slips. **Cite the derived form in your manuscript, not eq. (6) or eq. (17) verbatim.**

### 1.4 The paper's own ablation (Fig. 2d)

Murray tests the two approximations separately:

- **Local only** (locality approximation, symmetric feedback $W^{out\top}$): performs like full RFLO.
- **RF only** (exact non-local gradient, random feedback $B$): performs like BPTT.

So almost all of the gap between RFLO and BPTT comes from **dropping the non-local term**, not from the random feedback. This matters for §5 below.

---

## 2. Paper vs. the authors' notebook (`RFLO.ipynb`)

| Item | Paper (eqs. 1, 5, 6; Methods) | `RFLO.ipynb` | Verdict |
|---|---|---|---|
| Dynamics | $h(t)=h(t-1)+\frac{1}{\tau}[-h(t-1)+\phi(u(t))]$ | `h[tt+1] = h[tt] + (-h[tt] + f(u[tt+1]))/tau_m` | Same |
| Total input | $u(t)=Wh(t-1)+W^{in}x(t)$ | `u[tt+1] = w_rec@h[tt] + w_in@x[tt+1]` | Same |
| Output / error | $y=W^{out}h,\;\varepsilon=y^*-y$ | `y[tt+1] = w_out@h[tt+1]`; `err = y_ - y` | Same |
| $\phi'$ | $1-\tanh^2$ | `1/cosh(10*tanh(x/10))**2` (overflow-safe sech²) | Equivalent |
| Recurrent trace $p$ | $\phi'(u_a(t))\,h_b(t-1)$ | `outer(df(u[tt+1]), h[tt])` | **Correct** |
| Input trace $q$ | $\phi'(u_a(t))\,x_b(t)$ (derived) | `outer(df(u[tt+1]), x[tt])`, i.e. $x(t-1)$ | **One-step lag** (copies eq. 6). The RTRL branch uses `x[tt+1]`, which is correct. No effect in the demo: $x\equiv0$ and $\eta_{in}=0$. |
| $W^{out}$ update | $\eta_1\varepsilon h^{\top}$ | `eta1/t_max*outer(err, h)` | Same, scaled by $1/T$ |
| $W$, $W^{in}$ updates | $\eta[B\varepsilon]_a p_{ab}$ | `eta2*outer(b@err, ones)*p/t_max` | Same, scaled by $1/T$ |
| Update timing | online, or batched per trial | `online_learning` flag (default: per trial) | Same |
| Init $W$ | $\mathcal{N}(0, g^2/N)$, $g=1.5$ | `1.5*randn/sqrt(N)` | Same |
| Init $W^{in}$ | $\mathcal{U}[-1,1]$ | `0.1*(rand-1)`, i.e. $\mathcal{U}[-0.1,0]$ | Differs (sign-biased, small) |
| Init $W^{out}$ | $\mathcal{U}[-1/\sqrt N, 1/\sqrt N]$ | `0.1*(2*rand-1)/sqrt(N)` | 10× smaller |
| Feedback $B$ | $\mathcal{N}(0,1)$ | `randn/sqrt(n_out)` | Same for $n_{out}=1$ |
| Learning-rate order | $\eta_1$ = out, $\eta_2$ = rec, $\eta_3$ = in | list is `[eta_in, eta_rec, eta_out]` | Convention only |
| Bias | none | none | Same |

**Conclusion:** the notebook implements RFLO faithfully. The only formula deviation is the one-step lag in the input trace $q$, inherited from the paper's eq. (6), and it is inactive in the demo.

---

## 3. What `DRTLO_GD_R2.py` implements

A **stack of $L$ leaky recurrent layers**, trained online, one sample per `fit()` call. Notation: layer $l=1..L$; $h^0(t) \equiv x(t)$; $\tilde v = [v;\,1]$ appends the bias input.

$$
u^l(t) = W^l\,h^l(t-1) + U^l\,\tilde h^{\,l-1}(t), \qquad
h^l(t) = \Big(1-\tfrac{1}{\tau}\Big)h^l(t-1) + \tfrac{1}{\tau}\,\phi\big(u^l(t)\big), \qquad
y(t) = W^{out} h^L(t)
$$

In the code: $W^l$ = `wR[l]`, $U^l$ = `wI[l]`, $W^{out}$ = `wO`, $B^l$ = `BS[l]`; $\eta_1,\eta_2,\eta_3$ = `ηS` (out, rec, in), the same order as the paper.

### 3.1 Side-by-side

| Step | Murray RFLO (1 layer) | `DRTLO_GD_R2.fit` (layer $l$) | Lines | Verdict |
|---|---|---|---|---|
| Total input | $u=Wh(t-1)+W^{in}x(t)$ | $u^l=W^l h^l(t-1)+U^l\tilde h^{l-1}(t)$ | 102 | Same (adds bias, stacks layers) |
| State | $h(t)=(1-\frac1\tau)h(t-1)+\frac1\tau\phi(u(t))$ | identical | 103 | Same |
| Output | $y=W^{out}h(t)$ | $y=W^{out}h^L(t)$ | 109 | Same |
| Error | $\varepsilon=y^*-y$ | `eS = yR - yP` | 110 | Same |
| Recurrent trace | $\frac1\tau\phi'(u_a(\mathbf{t}))h_b(t-1)+(1-\frac1\tau)p$ | $\frac1\tau\phi'(u^l_a(\mathbf{t-1}))h^l_b(t-1)+(1-\frac1\tau)p$ | 115 | **Wrong time index** (E1) |
| Input trace | $\frac1\tau\phi'(u_a(\mathbf{t}))x_b(t)+(1-\frac1\tau)q$ | $\frac1\tau\phi'(u^l_a(\mathbf{t-1}))\tilde h^{l-1}_b(t)+(1-\frac1\tau)q$ | 116 | **Wrong time index** (E1) |
| Error routed to the hidden layer | $B\varepsilon$, with $B\in\mathbb{R}^{N\times n_{out}}$ | $B^l\varepsilon$, a separate random $B^l$ for **every** layer | 60, 118–119 | Still random feedback (E2). Deep version = Direct Feedback Alignment (E3) |
| $W$ update | $\eta_2[B\varepsilon]_a p_{ab}$ | `η2*outer(BS[l]@eS, ones)*pS[l]` | 118, 122 | Same form |
| $W^{in}$ update | $\eta_3[B\varepsilon]_a q_{ab}$ | `η3*outer(BS[l]@eS, ones)*qS[l]` | 119, 121 | Same form |
| $W^{out}$ update | $\eta_1\varepsilon h^{\top}$ (exact gradient) | `η1*outer(eS, h_L)` | 130–131 | Same |
| Update timing | per step or per trial (÷T) | every step, no ÷T | — | Fine for a data stream |
| Sign | gradient descent | `+=` with $\varepsilon=y^*-y$ | — | Correct |
| Init $W$ | $\mathcal N(0,1.5^2/N)$ | Xavier-uniform, variance $\approx 1/N$ (i.e. $g\approx1$) | 54 | Differs (E6) |
| Init $B$ | $\mathcal N(0,1)$ | Xavier-uniform, small | 60 | Differs (E6) |
| Seeds | random per network | `np.random.seed(42)` plus fixed `sd` per matrix | 6, 48–60 | Every granule is identical at birth (E4) |

**Everything except the trace timing (E1) matches RFLO's update form exactly.** The deep extension (feedback sent to each layer) goes beyond what the paper covers; see E3.

---

## 4. Findings

### E1 — Eligibility traces use $\phi'(u(t-1))$ instead of $\phi'(u(t))$ · **bug** · lines 115–116

`self.uP[l]` still holds the previous step's input when the traces are updated. `new_uP[l]`, the current one, has already been computed at line 104 but is not used.

- In $p$, both factors are lagged. This is internally consistent with the paper's eq. (17), but it does not match the derivative of the model.
- In $q$, $\phi'$ is lagged while the input is current, so **the two factors refer to different time steps**. This matches none of the paper's variants.
- On the very first call, `uP = 0`, so $\phi'(0)=1$ is used regardless of the actual input.

**Numerical check.** With the real `RTLO` class (1 layer, 6 units, τ = 10), I compared the term the code adds to $p$ against a central finite difference of $h(t)$ with respect to $W_{ab}$, holding $h(t-1)$ fixed:

| | value | relative error |
|---|---|---|
| finite difference (true local derivative) | −8.867e−03 | — |
| code, $\phi'(u(t-1))$ | −4.063e−03 | **54 %** |
| corrected, $\phi'(u(t))$ | −8.867e−03 | 0.00 % |

**Fix:** use `dActivation(new_uP[l], self.act)` in lines 115–116.

### E2 — The random feedback was *not* removed · **contradicts the intended design** · lines 60, 118–119

`self.BS[l]` is a fixed random matrix (never updated), and it multiplies the error in both hidden-layer updates. So the file is still **random-feedback** learning, despite the "GD" in its name.

The version that sends the error back through the real weights lives in `DRTLO_QN_R1.py` / `DRTLO_QN_R2.py` (`wO.T @ eS`, then `wI.T @ du`). R07 does not use those files.

### E3 — The deep extension is Direct Feedback Alignment, which Murray's paper does not cover · **theory gap, not a bug**

Each hidden layer receives $B^l\varepsilon$ directly from the output. That is *Direct Feedback Alignment* (Nøkland, 2016) combined with RFLO's eligibility traces.

Murray's convergence argument (Appendix 2: the loss decreases when $W^{out}$ aligns with $B$) is derived for **one** linear layer only. For $L>1$:

- the lower layers' feedback $B^l$ has to align with the *effective* path from layer $l$ to the output, and nothing in the paper guarantees that;
- the manuscript has to state this explicitly and cite Nøkland (2016), or switch to option (a) in §5.

This links directly to Reviewer 1, point 3 (no convergence or stability analysis) and Reviewer 4, point 1 (novelty).

### E4 — Fixed seeds and a global `np.random.seed(42)` · **design issue** · lines 6, 48–60

- Every `RTLO` starts with **identical** weights and identical feedback matrices, because the seeds 39–42 are hard-coded. A new granule therefore starts as an untrained clone of the very first granule's initial state. That weakens the "granule-specific model" argument and matches the identical weights seen in R07 cells 9–10.
- Calling `np.random.seed(42)` inside the constructor resets NumPy's **global** generator each time a granule is created. That silently affects any other code that draws from `np.random`.

Suggested fix: pass a `numpy.random.Generator` (or a per-granule seed) into the constructor and never reseed the global generator.

### E5 — The full weight history is stored at every step · **memory and complexity** · lines 124–125, 132

`wR_hist`, `wI_hist` and `wO_hist` append flattened copies of every matrix on every `fit`. Memory grows as $\mathcal O(\text{steps}\times\text{parameters}\times\text{granules})$. That inflates run time during the Optuna search, and it would distort any computational-cost figure reported in answer to Reviewer 1, point 1. Make it optional (e.g. `store_hist=False`).

### E6 — Initialisation differs from the paper · **document it** · lines 48–60

- **Recurrent weights:** Xavier-uniform gives variance ≈ $1/N$ ($g≈1$, edge of chaos), against Murray's $g=1.5$ (chaotic regime, richer dynamics).
- **Feedback:** Xavier-uniform $B$ is small, against Murray's $\mathcal N(0,1)$. A smaller $B$ scales down the hidden-layer learning, so it is effectively coupled to $\eta_2,\eta_3$.

Neither is wrong, but the paper should state the choice. Both affect how fast feedback alignment happens.

### E7 — Dead state · **cleanup** · various lines

`j`, `start`, `ref`, `ρ`, `rR`/`rP`/`rL`/`rU`, `rRsum`, `yP_hist`, `uP2`, `εY…`, `rulR…` are never used. `fit()` also sets `self.n` (line 92), which only matters for prediction. Removing these would make the class easier to describe in the paper, which helps with Reviewer 1, point 7 (reproducibility).

### What is coherent

- Within a time step, the forward pass in `fit` is identical to the one in `PredictSingle`/`PredictIntr`. AutoCloud calls `Restore()` before `Adapt()`, so the state used for the prediction and the state used for learning are the same. **No look-ahead.**
- Trace shapes, including the bias column, are right for every layer: `qS[0]` is $N_1\times(n_I+1)$ and `qS[l]` is $N_l\times(N_{l-1}+1)$.
- The trace at time $t$ is updated before it is multiplied by $\varepsilon(t)$, as in eq. (5).
- Per step, the cost is $\mathcal O\big(\sum_l N_l^2 + N_l N_{l-1}\big)$, the same order as RFLO and BPTT.

### Integration notes (in AutoCloud, but they affect the RNN)

- `AutoCloud.Adapt` calls `fit` only for granules with $\alpha_i>0$. Inactive granules keep a **frozen** hidden state $h(t)$ and frozen traces $p, q$. When such a granule is reactivated, its state belongs to some earlier time. Consider advancing $h$ for every granule (a forward pass is cheap) and training only the active ones.
- `AutoCloud.MergeCloudsRNN` is unreachable while `wtaG=True`, and broken if it ever runs (as reported earlier).

---

## 5. What option (a) — "remove the random feedback" — would look like

Goal: keep RFLO's local traces over time, but replace each random $B^l$ with the **true error signal within the current time step**.

Define the learning signal $\boldsymbol{\ell}^l(t) \approx \partial L(t)/\partial h^l(t)$, keeping only paths that stay inside time step $t$:

$$
\boldsymbol{\ell}^L(t) = (W^{out})^{\top}\varepsilon(t)
$$

$$
\boldsymbol{\ell}^{\,l}(t) = \frac{1}{\tau}\,\big(\bar U^{\,l+1}\big)^{\top}\Big[\phi'\big(u^{l+1}(t)\big)\odot \boldsymbol{\ell}^{\,l+1}(t)\Big], \qquad l = L-1,\dots,1
$$

Here $\bar U^{\,l+1}$ is `wI[l+1][:, :-1]`: the bias column is excluded because the bias does not depend on $h^l$. The updates keep exactly the same form; only $B^l\varepsilon$ is replaced by $\boldsymbol\ell^l$:

$$
\delta W^l_{ab} = \eta_2\,\ell^l_a(t)\,p^l_{ab}(t), \qquad
\delta U^l_{ab} = \eta_3\,\ell^l_a(t)\,q^l_{ab}(t), \qquad
\delta W^{out}_{ab} = \eta_1\,\varepsilon_a(t)\,h^L_b(t)
$$

with $p^l, q^l$ computed with $\phi'(u^l(t))$ (fix E1 first).

| | Current (`DRTLO_GD_R2`) | Option (a) |
|---|---|---|
| Error to layer $l$ | $B^l\varepsilon$ (fixed random) | $\boldsymbol\ell^l$ (true, within one step) |
| Feedback matrices to tune / seed | $L$ | none |
| Theory | Murray App. 2 (1 layer) + DFA (heuristic) | Murray's *"Local only"* variant (Fig. 2d), extended across layers; same family as *e-prop* (Bellec et al., 2020) |
| Remaining approximation | locality **and** random feedback | locality only |
| Cost per step | $\mathcal O(\sum N_l^2)$ | $\mathcal O(\sum N_l^2)$ (one extra matrix–vector product per layer) |
| Online / causal | yes | yes |

**Expected effect:** according to Murray's Fig. 2d, removing the random feedback alone gives a **small** accuracy gain, because locality causes most of the gap to BPTT. The main benefits of (a) are therefore:

- **Fewer arbitrary components:** no random $B^l$ and no seeds to justify.
- **A cleaner theory statement:** an exact gradient with respect to the current state, truncated in time. This is easier to defend against Reviewer 1, point 3 and Reviewer 4, point 1.
- **A ready-made ablation** (RFLO-DFA vs. RFLO-symmetric), which answers Reviewer 2, point 7 and Reviewer 4, point 2.

---

## 6. Summary

| # | Finding | Type | Where |
|---|---|---|---|
| E1 | Traces use $\phi'(u(t-1))$; must be $\phi'(u(t))$ (54 % error in the check) | **Bug** | `DRTLO_GD_R2.py:115-116` |
| E2 | Random feedback still present; the gradient version is only in the QN files | Intent mismatch | `:60`, `:118-119` |
| E3 | Deep extension = DFA, outside Murray's analysis | Theory gap | design |
| E4 | Identical init for every granule; global reseed | Design | `:6`, `:48-60` |
| E5 | Weight history stored every step | Cost | `:124-125`, `:132` |
| E6 | Init differs from the paper ($g$, $B$ scale) | Document | `:48-60` |
| E7 | Unused attributes | Cleanup | various |
| — | Paper eqs. (6)/(11)/(17) disagree on time indices; use the derived form | Writing | rflo.pdf |
| — | Authors' notebook lags $x$ in $q$ (inactive in their demo) | Reference code | RFLO.ipynb |

No code was changed.
